# TCC - Análise Preditiva de Risco Acadêmico Escolar
## CRISP-DM: 6. Deployment


## O que significa deployment neste projeto

A pipeline gera sinais técnicos: nota prevista, risco previsto, erro, faixa de risco e fatores de alerta. O deployment transforma esses sinais em produtos de uso escolar.

Neste projeto, deployment não significa publicar um modelo em produção aberta na internet. Significa transformar a saída analítica em relatórios operacionais, rankings e visualizações que possam ser usados internamente pela escola.

Produtos entregues:

- relatório do professor;
- relatório da coordenação;
- relatório da secretaria;
- ranking executivo de alunos prioritários;
- rankings de professores com maior concentração de alunos em risco;
- dashboard Streamlit para visualização gráfica.


## Carregar bibliotecas e localizar o projeto

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "school_predictor").exists() and (candidate / "artifacts").exists():
            return candidate.resolve()
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## Importar pontos de deployment

A entrega operacional usa a pipeline completa e a geração de relatórios finais.



In [ ]:
from school_predictor.project import ProjectPaths
from school_predictor.pipeline.orchestration import run_full_reporting_pipeline
from school_predictor.pipeline.reporting import build_school_reports

## Comandos operacionais

Na arquitetura atual, existem dois níveis de operação:

1. ciclo privado com banco local, usado por quem possui acesso institucional;
2. ciclo público da pipeline, que parte dos CSVs canônicos já extraídos.

Se for necessário preparar um banco restaurado e depois extrair os CSVs:

```bash
./.venv/bin/python -m school_predictor prepare-db
./.venv/bin/python -m school_predictor extract --project-root .
```

A execução principal da pipeline pública acontece pela CLI:

```bash
./.venv/bin/python -m school_predictor workflow --project-root .
```

Esse comando parte dos CSVs canônicos já disponíveis, roda os dois modos técnicos (`previsao_nota` e `alerta_risco`) e depois gera os relatórios finais.

Também é possível rodar apenas os relatórios, caso os artefatos da modelagem já existam:

```bash
./.venv/bin/python -m school_predictor reports --project-root .
```

Para abrir o dashboard:

```bash
./.venv/bin/streamlit run dashboard_streamlit.py
```




### Exemplo fixo do que cada comando gera

Abaixo está um resumo simples do papel operacional de cada comando principal:

<div style="font-size: 1em; overflow-x: auto;">

| comando | gera |
| --- | --- |
| python -m school_predictor extract | CSVs canônicos anonimizados a partir do banco tratado |
| python -m school_predictor pipeline --mode previsao_nota | artefatos técnicos da modelagem de previsão de nota |
| python -m school_predictor pipeline --mode alerta_risco | artefatos técnicos da modelagem de alerta de risco |
| python -m school_predictor reports | relatórios finais e rankings de uso institucional |
| python -m school_predictor workflow | execução encadeada da pipeline com relatórios finais |

</div>


## Opcional: executar deployment localmente

Por padrão, este notebook não executa a pipeline nem sobrescreve arquivos. Para gerar relatórios localmente, altere uma das flags abaixo.

`RUN_FULL_WORKFLOW` parte dos CSVs canônicos e recalcula modelagem + relatórios. `RUN_REPORTS_ONLY` apenas reconstrói a camada de deployment a partir dos artefatos de `previsao_nota` e `alerta_risco`.


In [ ]:
RUN_FULL_WORKFLOW = False
RUN_REPORTS_ONLY = False

if RUN_FULL_WORKFLOW:
    run_full_reporting_pipeline(project_root=PROJECT_ROOT)
elif RUN_REPORTS_ONLY:
    build_school_reports(project_root=PROJECT_ROOT)
else:
    print("Nenhuma execucao realizada. Altere RUN_FULL_WORKFLOW ou RUN_REPORTS_ONLY para True para rodar localmente.")

### O que acontece quando o deployment roda

Se você ativar uma das flags, o projeto deixa de estar apenas no nível analítico e passa a gerar saídas operacionais em `artifacts/reports/`. Ou seja, o dado que antes estava espalhado entre artefatos técnicos vira produto consumível por perfis da escola.



## Artefatos entregues

Os relatórios finais ficam em `artifacts/reports/`. Eles são derivados dos resultados de `artifacts/pipeline/previsao_nota/` e `artifacts/pipeline/alerta_risco/`.

A função `build_school_reports` primeiro cruza os dois modos em uma base integrada e, a partir dela, deriva visões específicas para professor, coordenação, secretaria e ranking executivo.


### Exemplo fixo dos artefatos finais

Estes são os principais arquivos entregues ao final do deployment:

<div style="font-size: 1em; overflow-x: auto;">

| arquivo | destinatário | uso |
| --- | --- | --- |
| relatorio_professor.csv | professor | ação pedagógica por aluno e disciplina |
| relatorio_coordenacao.csv | coordenação | priorização por série e disciplina |
| relatorio_secretaria.csv | secretaria | apoio administrativo e contato com a família |
| ranking_executivo_coordenacao.csv | coordenação | fila executiva dos casos mais críticos |
| relatorio_professores_por_turma.csv | coordenação | ranking de professores por turma |
| relatorio_professores_geral.csv | coordenação | ranking consolidado de professores |

</div>


In [ ]:
project_paths = ProjectPaths.from_root(PROJECT_ROOT)
REPORT_DIR = project_paths.reports_artifacts_dir

expected_reports = {
    "relatorio_escolar_integrado.csv": "base operacional integrada com nota prevista, risco, fatores de alerta e recomendacoes",
    "relatorio_professor.csv": "visao por aluno, disciplina e etapa para apoio ao professor",
    "relatorio_professor.txt": "versao textual resumida do relatorio do professor",
    "relatorio_coordenacao.csv": "agrupamentos por periodo, serie, disciplina e professor para coordenacao",
    "relatorio_coordenacao.txt": "versao textual resumida do relatorio da coordenacao",
    "relatorio_secretaria.csv": "visao administrativa por aluno para acompanhamento de frequencia, financeiro e risco",
    "relatorio_secretaria.txt": "versao textual resumida do relatorio da secretaria",
    "ranking_executivo_coordenacao.csv": "ranking de alunos prioritarios para acao da coordenacao",
    "ranking_executivo_coordenacao.txt": "versao textual resumida do ranking executivo",
    "relatorio_professores_por_turma.csv": "quantidade de alunos por nivel de risco considerando curso, serie, turma e professor",
    "relatorio_professores_geral.csv": "quantidade de alunos por nivel de risco por professor independentemente da turma",
}

def count_output_rows(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as file:
        lines = sum(1 for _ in file)
    if path.suffix.lower() == ".csv":
        return max(lines - 1, 0)
    return lines


report_inventory = []
for filename, description in expected_reports.items():
    path = REPORT_DIR / filename
    report_inventory.append({
        "arquivo": filename,
        "existe": path.exists(),
        "linhas": count_output_rows(path),
        "tipo": path.suffix.lower().lstrip("."),
        "descricao": description,
    })

pd.DataFrame(report_inventory)

### O que este inventário comprova

A tabela acima mostra se a camada final de entrega realmente foi materializada. Quando os arquivos aparecem, o deployment deixa de ser conceito e passa a ser evidência: existem relatórios, rankings e bases prontas para uso.

Também fica claro que esta fase não gera um único arquivo. Ela gera uma família de saídas, cada uma com um papel institucional diferente.



## Leitura segura dos relatórios

As amostras abaixo evitam identificadores diretos sempre que possível. A finalidade é mostrar a forma do deployment, não expor dados pessoais.

Como o deployment trabalha sobre a base integrada, as visualizações a seguir mostram apenas recortes seguros das saídas finais.



In [ ]:
def public_read_report(filename: str, columns: list[str] | None = None, rows: int = 5) -> pd.DataFrame:
    path = REPORT_DIR / filename
    if not path.exists():
        print(f"Arquivo ausente: {path}")
        return pd.DataFrame()
    frame = pd.read_csv(path, encoding="utf-8", low_memory=False)
    if columns is not None:
        frame = frame[[column for column in columns if column in frame.columns]]
    return frame.head(rows)


teacher_columns = [
    "NomeSerie", "NomeDisciplina", "NomeEtapa", "professor_responsavel",
    "nota_atual", "nota_prevista_proxima", "pontuacao_risco", "nivel_risco",
    "fatores_alerta", "recomendacao_professor",
]

public_read_report("relatorio_professor.csv", teacher_columns)

## Amostra segura da base integrada do deployment

Antes de separar por perfil, vale ver um recorte da base integrada que alimenta toda a camada final. Ela cruza os dois modos técnicos e concentra nota atual, previsão, risco, professor e fatores de alerta em uma mesma observação.


### Exemplo fixo da base integrada

Esta amostra mostra a visão consolidada que sustenta a priorização executiva, já usando os nomes pseudonimizados da base pública:

<div style="font-size: 1em; overflow-x: auto;">

| NomeAluno | NomePeriodo | NomeSerie | casos_alto_risco | indice_prioridade | nivel_prioridade_executiva | motivos_prioridade |
| --- | --- | --- | --- | --- | --- | --- |
| Aluno 01560 | 2025 | 3ª Série | 28 | 113 | crítica | volume muito alto de disciplinas em alto risco; há disciplinas com gravidade máxima muito elevada; faltas acumuladas em faixa de alerta; pendências financeiras recorrentes |
| Aluno 04929 | 2025 | 1ª Série | 29 | 112 | crítica | volume muito alto de disciplinas em alto risco; há disciplinas com gravidade máxima muito elevada; faltas acumuladas em faixa crítica; média prevista abaixo da linha de segurança |
| Aluno 01742 | 2025 | 3ª Série | 28 | 107 | crítica | volume muito alto de disciplinas em alto risco; média prevista abaixo da linha de segurança |
| Aluno 03272 | 2025 | 3ª Série | 26 | 102 | crítica | volume muito alto de disciplinas em alto risco; média prevista abaixo da linha de segurança |
| Aluno 00876 | 2025 | 1ª Série | 22 | 98 | crítica | volume muito alto de disciplinas em alto risco; há disciplinas com gravidade máxima muito elevada; faltas acumuladas em faixa crítica; média prevista abaixo da linha de segurança |

</div>



In [ ]:
integrated_columns = [
    "NomePeriodo", "NomeSerie", "NomeDisciplina", "NomeEtapa",
    "professor_responsavel", "ValorMedia", "predicted_next_grade",
    "predicted_risk_flag", "pontuacao_risco", "nivel_risco", "fatores_alerta",
]

public_read_report("relatorio_escolar_integrado.csv", integrated_columns)

### O que a base integrada representa

Quando esta amostra aparece, você consegue ver o ponto em que o projeto deixa de ter artefatos separados de `previsao_nota` e `alerta_risco` e passa a ter uma visão operacional unificada.

Essa base é o elo entre modelagem e uso institucional. Sem ela, os relatórios finais ficariam fragmentados.



### O que a amostra do relatório do professor mostra

Depois da leitura acima, o deployment já se traduz em ação pedagógica. O professor deixa de ver apenas predições e passa a ver casos priorizados com recomendação textual.


## Entrega para o professor

O relatório do professor responde: quais alunos precisam de atenção, em qual disciplina, qual a nota atual, qual a nota prevista, qual o nível de risco e qual ação pedagógica é sugerida.



### Exemplo fixo do relatório do professor

Esta é uma pequena amostra do que o professor recebe para agir em sala, já com os nomes pseudonimizados da base pública:

<div style="font-size: 1em; overflow-x: auto;">

| NomePeriodo | NomeSerie | NomeDisciplina | NomeAluno | nivel_risco | nota_atual | nota_prevista_proxima | recomendacao_professor |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | 9º Ano | Matemática 2 | Aluno 04819 | alto | 3.8 | 2.494 | Intervenção imediata. Motivos: nota prevista abaixo da média de aprovação; tendência recente de queda; faltas acumuladas elevadas; nota atual já abaixo do esperado. Ação pedagógica: Priorizar retomada de pré-requisitos, resolução guiada de exercícios e verificação de lacunas conceituais. |
| 2025 | 9º Ano | Matemática 2 | Aluno 03877 | alto | 4.0 | 3.4718 | Intervenção imediata. Motivos: nota prevista abaixo da média de aprovação; tendência recente de queda; faltas acumuladas elevadas; pendências financeiras recorrentes; nota atual já abaixo do esperado. Ação pedagógica: Priorizar retomada de pré-requisitos, resolução guiada de exercícios e verificação de lacunas conceituais. |
| 2025 | 8º Ano | Matemática 1 | Aluno 00505 | alto | 4.6 | 4.9705 | Intervenção imediata. Motivos: nota prevista abaixo da média de aprovação; faltas acumuladas elevadas; pendências financeiras recorrentes; nota atual já abaixo do esperado. Ação pedagógica: Priorizar retomada de pré-requisitos, resolução guiada de exercícios e verificação de lacunas conceituais. |
| 2025 | 8º Ano | Matemática 2 | Aluno 00505 | alto | 2.7 | 2.6337 | Intervenção imediata. Motivos: nota prevista abaixo da média de aprovação; faltas acumuladas elevadas; pendências financeiras recorrentes; nota atual já abaixo do esperado. Ação pedagógica: Priorizar retomada de pré-requisitos, resolução guiada de exercícios e verificação de lacunas conceituais. |
| 2025 | 1ª Série | Literatura | Aluno 03078 | alto | 3.3 | 2.8276 | Intervenção imediata. Motivos: nota prevista abaixo da média de aprovação; faltas acumuladas elevadas; pendências financeiras recorrentes; nota atual já abaixo do esperado. Ação pedagógica: Trabalhar leitura orientada, interpretação, produção textual curta e devolutiva individual frequente. |

</div>



In [ ]:
teacher = pd.read_csv(REPORT_DIR / "relatorio_professor.csv", encoding="utf-8", low_memory=False)

teacher_summary = (
    teacher.groupby(["NomeDisciplina", "nivel_risco"], dropna=False)
    .size()
    .reset_index(name="quantidade")
    .sort_values(["nivel_risco", "quantidade"], ascending=[True, False], kind="stable")
)

teacher_summary.head(15)

### Como interpretar o resumo do professor

A tabela agrupada mostra concentração de risco por disciplina. Isso permite que o deployment não sirva apenas para ler aluno por aluno, mas também para perceber onde uma disciplina inteira pode merecer revisão de estratégia.


## Entrega para a coordenação

A coordenação recebe agrupamentos por série, disciplina, turma e professor, além de ranking executivo para priorizar intervenções.

Nessa camada, o foco já não é mais uma única linha por aluno e etapa, mas uma visão agregada para decidir onde intervir primeiro.



### Exemplo fixo do relatório da coordenação

Aqui a coordenação passa a ver agrupamentos críticos por série e disciplina:

<div style="font-size: 0.9em; overflow-x: auto;">

| NomePeriodo | NomeSerie | NomeDisciplina | alunos_monitorados | casos_alto_risco | taxa_alto_risco | grupos_sem_professor_identificado | recomendacao_coordenacao |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | 9º Ano | Matemática 2 | 109 | 109 | 1.0 | 1 | Planejar intervenção coletiva na disciplina, revisar estratégia docente e acompanhar turma em curto prazo. |
| 2025 | 1ª Série | Literatura | 97 | 97 | 1.0 | 0 | Planejar intervenção coletiva na disciplina, revisar estratégia docente e acompanhar turma em curto prazo. |
| 2025 | 1ª Série | Geografia 2 | 97 | 97 | 1.0 | 0 | Planejar intervenção coletiva na disciplina, revisar estratégia docente e acompanhar turma em curto prazo. |
| 2025 | 1ª Série | Química 2 | 97 | 97 | 1.0 | 0 | Planejar intervenção coletiva na disciplina, revisar estratégia docente e acompanhar turma em curto prazo. |
| 2025 | 7º Ano | Matemática 2 | 96 | 96 | 1.0 | 0 | Planejar intervenção coletiva na disciplina, revisar estratégia docente e acompanhar turma em curto prazo. |

</div>


In [ ]:
coordination_columns = [
    "NomePeriodo", "NomeSerie", "NomeDisciplina", "professor_responsavel",
    "alunos_monitorados", "casos_alto_risco", "casos_risco_moderado",
    "taxa_alto_risco", "recomendacao_coordenacao",
]

public_read_report("relatorio_coordenacao.csv", coordination_columns)

### O que a coordenação passa a receber

Nesta saída, o dado deixa de ser individual e passa a ser agregado para decisão gerencial. A coordenação ganha visão de volume, taxa de alto risco e recomendação por agrupamento, o que facilita priorizar turmas, séries e disciplinas.


## Ranking de professores por risco

Esses dois relatórios ajudam a coordenação a enxergar concentração de risco por professor, com e sem recorte por turma.


### Exemplo fixo do ranking por turma

Para fins didáticos, a amostra abaixo mostra professores identificados, sem esconder que também existem casos sem vínculo docente no relatório real:

<div style="font-size: 1em; overflow-x: auto;">

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | professor_responsavel | casos_alto_risco | casos_risco_moderado | casos_baixo_risco | taxa_alunos_com_risco |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Médio | 1ª Série | 1ª Série A - Matutino | Professor 00077 | 50 | 0 | 0 | 1.0 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série A - Matutino | Professor 00131 | 50 | 0 | 0 | 1.0 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série A - Matutino | Professor 00192 | 50 | 0 | 0 | 1.0 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série A - Matutino | Professor 00061 | 44 | 6 | 0 | 1.0 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | Professor 00077 | 47 | 0 | 0 | 1.0 |

</div>

### Exemplo fixo do ranking geral

<div style="font-size: 1em; overflow-x: auto;">

| NomePeriodo | professor_responsavel | alunos_monitorados | casos_alto_risco | casos_risco_moderado | casos_baixo_risco | taxa_alunos_com_risco |
| --- | --- | --- | --- | --- | --- | --- |
| 2025 | Professor 00131 | 368 | 229 | 81 | 58 | 0.8424 |
| 2025 | Professor 00192 | 369 | 143 | 121 | 105 | 0.7154 |
| 2025 | Professor 00087 | 330 | 215 | 35 | 80 | 0.7576 |
| 2025 | Professor 00061 | 260 | 200 | 46 | 14 | 0.9462 |
| 2025 | Professor 00076 | 384 | 176 | 68 | 140 | 0.6354 |

</div>


In [ ]:
teacher_by_class_columns = [
    "NomeCurso", "NomeSerie", "ApelidoTurma", "professor_responsavel",
    "alunos_com_risco", "casos_alto_risco", "casos_risco_moderado", "casos_baixo_risco",
]
teacher_overall_columns = [
    "professor_responsavel", "alunos_com_risco", "casos_alto_risco",
    "casos_risco_moderado", "casos_baixo_risco",
]

display(public_read_report("relatorio_professores_por_turma.csv", teacher_by_class_columns))
display(public_read_report("relatorio_professores_geral.csv", teacher_overall_columns))

### O que os rankings de professores mostram

Esses dois relatórios mudam a escala da leitura. Em vez de olhar apenas para disciplinas ou alunos, a coordenação consegue enxergar concentração de risco vinculada aos docentes, com e sem recorte por turma.


## Entrega para a secretaria

A secretaria recebe uma visão administrativa, com alertas de frequência, financeiro e necessidade de acompanhamento.

Esse relatório não substitui a coordenação pedagógica; ele complementa a operação com sinais administrativos que podem influenciar o acompanhamento do aluno.


### Exemplo fixo do relatório da secretaria

A secretaria recebe uma visão focada em frequência, financeiro e prioridade de contato, já com os nomes pseudonimizados da base pública:

<div style="font-size: 1.0em; overflow-x: auto;">

| NomeAluno | NomePeriodo | NomeSerie | casos_alto_risco | faltas_acumuladas_maximas | pagamentos_pendentes_maximos | prioridade_secretaria | recomendacao_secretaria |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Aluno 01560 | 2025 | 3ª Série | 28 | 9.0 | 5.0 | alta | Validar histórico financeiro e registrar contato preventivo; confirmar frequência e alinhar comunicação com a família; avisar coordenação sobre volume de disciplinas em risco. |
| Aluno 06423 | 2025 | 3ª Série | 18 | 16.0 | 2.0 | alta | Validar histórico financeiro e registrar contato preventivo; confirmar frequência e alinhar comunicação com a família; avisar coordenação sobre volume de disciplinas em risco. |
| Aluno 00618 | 2025 | 2ª Série | 9 | 11.0 | 12.0 | alta | Validar histórico financeiro e registrar contato preventivo; confirmar frequência e alinhar comunicação com a família; avisar coordenação sobre volume de disciplinas em risco. |
| Aluno 05327 | 2025 | 2ª Série | 9 | 11.0 | 12.0 | alta | Validar histórico financeiro e registrar contato preventivo; confirmar frequência e alinhar comunicação com a família; avisar coordenação sobre volume de disciplinas em risco. |
| Aluno 03078 | 2025 | 1ª Série | 9 | 17.0 | 8.0 | alta | Validar histórico financeiro e registrar contato preventivo; confirmar frequência e alinhar comunicação com a família; avisar coordenação sobre volume de disciplinas em risco. |

</div>


In [ ]:
secretary_columns = [
    "NomePeriodo", "NomeSerie", "casos_alto_risco", "casos_risco_moderado",
    "alerta_frequencia", "alerta_financeiro", "recomendacao_secretaria",
]

public_read_report("relatorio_secretaria.csv", secretary_columns)

### O que a secretaria passa a enxergar

Aqui o deployment ganha camada administrativa. O dado já não fala apenas de aprendizagem: ele combina risco pedagógico com sinais de frequência e financeiro para orientar acompanhamento institucional.


## Dashboard Streamlit

O dashboard consome os CSVs em `artifacts/reports/` e apresenta visões gráficas para diferentes perfis. Ele não treina modelos; apenas visualiza as saídas já geradas.

Portanto, se os relatórios ainda não existirem, o dashboard não substitui a pipeline: ele depende dela.



### Exemplo fixo do que cada perfil consome

A tabela abaixo resume qual artefato atende cada perfil institucional:

<div style="font-size: 1em; overflow-x: auto;">

| perfil | consome | o_que_enxerga |
| --- | --- | --- |
| Professor | relatorio_professor.csv | alunos por disciplina, fatores de alerta e recomendação pedagógica |
| Coordenação | relatorio_coordenacao.csv + rankings | priorização por série, disciplina, turma e professor |
| Secretaria | relatorio_secretaria.csv | faltas, pendências e prioridade de contato administrativo |

</div>


In [ ]:
dashboard_entrypoint = PROJECT_ROOT / "dashboard_streamlit.py"
dashboard_module = PROJECT_ROOT / "school_predictor" / "app" / "dashboard.py"

pd.DataFrame([
    {"item": "entrypoint", "caminho": str(dashboard_entrypoint), "existe": dashboard_entrypoint.exists()},
    {"item": "modulo", "caminho": str(dashboard_module), "existe": dashboard_module.exists()},
    {"item": "comando", "caminho": "./.venv/bin/streamlit run dashboard_streamlit.py", "existe": True},
])

### O que o dashboard consome na prática

A tabela acima deixa claro que o Streamlit não substitui a pipeline. Ele consome os relatórios prontos e os transforma em leitura visual. Ou seja, o dashboard é a vitrine da saída final, não a origem do processamento.



## Critérios de sucesso do deployment

O deployment é considerado adequado quando:

- os relatórios são gerados sem depender de SQL ou credenciais durante a etapa pública de deployment;
- professor consegue priorizar alunos e disciplinas;
- coordenação consegue enxergar concentrações de risco;
- secretaria consegue acompanhar alertas administrativos;
- dashboard abre e carrega os CSVs finais;
- os dados continuam protegidos e fora do GitHub;
- a saída é explicável, com fatores de alerta e recomendações legíveis.




## Resultado da fase

Ao final do CRISP-DM, o projeto sai de dados escolares históricos e chega a uma entrega operacional:

- dados extraídos e anonimizados;
- dataset temporal preparado;
- modelos treinados e avaliados;
- base integrada de deployment cruzando previsão de nota e alerta de risco;
- predições e alertas gerados;
- relatórios finais por perfil em CSV e TXT;
- dashboard para leitura gráfica.

Essa etapa fecha o ciclo do TCC porque conecta o resultado técnico ao objetivo principal: apoiar professores, coordenadores e secretaria na identificação preventiva de risco acadêmico.



## Leitura didática do deployment

Ao final desta fase, o dado percorreu um caminho completo:
- nasceu como extração anonimizável;
- virou dataset temporal;
- foi transformado em predições e métricas;
- foi reunido em base integrada;
- terminou como relatório, ranking e dashboard.

Esse encadeamento didático ajuda a banca a entender não só o código, mas a trajetória do dado ao longo de todo o CRISP-DM.
